In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
raw_df = pd.read_csv('../input/spaceship-titanic/train.csv')
test_df = pd.read_csv('../input/spaceship-titanic/test.csv')

In [4]:
raw_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [5]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [6]:
categorical_cols_df = raw_df.select_dtypes(include=['object'])
numerical_cols_df = raw_df.select_dtypes(include=['float64'])
numerical_cols = list(numerical_cols_df.columns)
categorical_cols = list(categorical_cols_df.columns)

In [7]:
temp = dict(layout=go.Layout(font=dict(family='Times New Roman', size=13), width=800))

In [8]:
categorical_cols_df.drop(['PassengerId', 'Name', 'Cabin'], axis=1, inplace=True)

/opt/conda/lib/python3.7/site-packages/pandas/core/frame.py:4913: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


In [9]:
fig = px.histogram(categorical_cols_df, marginal='box')
fig.update_layout(template = temp, xaxis_title='Features Values')
fig.show()

In [10]:
fig = go.Figure()
for col in numerical_cols[1:]:
    fig.add_trace(go.Scatter(y=numerical_cols_df[col], name=col, mode='markers'))
    fig.update_xaxes(title='Index')
    fig.update_yaxes(title='Value')
fig.update_layout(template='plotly_dark', width=1200)
fig.show()

In [11]:
fig = px.scatter(raw_df, y='Age', color='Transported')
fig.update_layout(template='plotly_dark', width=1200)
fig.show()

In [12]:
n_cols = list(numerical_cols_df.columns)
c_cols = list(categorical_cols_df.columns)

In [13]:
numerical_columns_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', StandardScaler())
])

categorical_columns_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehotencoder', OneHotEncoder())
])

In [14]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_columns_transformer, n_cols),
    ('cat', categorical_columns_transformer, c_cols)
])

In [15]:
X_train = numerical_cols_df.join(categorical_cols_df)
Y_train = raw_df.iloc[:, -1]

In [16]:
x_train = preprocessor.fit_transform(X_train)
x_train

array([[ 0.71555276, -0.33310466, -0.28102673, ...,  1.        ,
         1.        ,  0.        ],
       [-0.32940751, -0.16807343, -0.27538657, ...,  1.        ,
         1.        ,  0.        ],
       [ 2.03916911, -0.2680006 ,  1.95999765, ...,  1.        ,
         0.        ,  1.        ],
       ...,
       [-0.19007947, -0.33310466, -0.28102673, ...,  1.        ,
         1.        ,  0.        ],
       [ 0.22790464, -0.33310466,  0.37636549, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.06387285, -0.14233462,  2.656871  , ...,  1.        ,
         1.        ,  0.        ]])

In [17]:
rf_model = RandomForestClassifier(random_state=42)

# Hyperparameter Tuning

In [18]:
parameters = {
    'n_estimators':[90, 100, 115, 130],
    'criterion':['gini', 'entropy'], 
    'max_depth':range(2,20,1),
    'min_samples_leaf':range(1, 10, 1),
    'min_samples_split':range(2,10,1),
    'max_features':['auto', 'log2']
}

In [19]:
clf =  RandomizedSearchCV(rf_model, parameters, cv=5)

In [20]:
clf.fit(x_train, Y_train)

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': range(2, 20),
                                        'max_features': ['auto', 'log2'],
                                        'min_samples_leaf': range(1, 10),
                                        'min_samples_split': range(2, 10),
                                        'n_estimators': [90, 100, 115, 130]})

In [21]:
print('Best Score : {}'.format(clf.best_score_))

Best Score : 0.7981155922712615


In [22]:
clf.best_params_

{'n_estimators': 130,
 'min_samples_split': 2,
 'min_samples_leaf': 3,
 'max_features': 'auto',
 'max_depth': 9,
 'criterion': 'gini'}

Almost same Identical patterns as in Training Data for Categorical Columns

In [23]:
id_col = test_df.iloc[:, 0]
test_df.drop(['PassengerId','Cabin', 'Name'], axis=1, inplace=True)
test_df = preprocessor.fit_transform(test_df)
y_predict = clf.predict(test_df)
df = {'PassengerId':id_col, 'Transported':y_predict}
final_df = pd.DataFrame(df)

In [24]:
final_df.to_csv('submission.csv', index=False)

# Thank You !